# Imports

In [ ]:
from config import Config
from data import ImagePairDataset
import matplotlib.pyplot as plt
from model import ImagePairMatcher
from train import EarlyStopper
from train import train, test
import torch
from torch import nn, optim
from torch.utils.data import DataLoader

# Data Setup

In [ ]:
train_data = ImagePairDataset(split='train')
validate_data = ImagePairDataset(split='validate')
test_data = ImagePairDataset(split='test')

train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=4, persistent_workers=True)
validate_loader = DataLoader(validate_data, batch_size=128, num_workers=4, persistent_workers=True)
test_loader = DataLoader(test_data, batch_size=128, num_workers=4, persistent_workers=True)

# Model Setup

In [ ]:
config = Config(CNN_DROPOUT=0.1, FC_DROPOUT=0.2, LEARNING_RATE=0.002)

In [ ]:
model = ImagePairMatcher.from_config(config)

In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total number of parameters: {total_params}')

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())
device = torch.device('cuda')
early_stop = EarlyStopper.from_config(config)

# Model Training

In [ ]:
train_results = train(model, criterion, optimizer, 50, train_loader, validate_loader, device, early_stop)

In [ ]:
plt.plot(train_results.train_losses, label='training')
plt.plot(train_results.validate_losses, label='validation')
plt.title('Training Losses')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

In [ ]:
plt.plot(train_results.train_accuracies, label='training')
plt.plot(train_results.validate_accuracies, label='validation')
plt.title('Training Accuracies')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Model Evaluation

In [ ]:
test_results = test(model, criterion, test_loader, device)

In [ ]:
print(f'(Loss) training: {round(train_results.train_losses[-1], 2)}, ' + 
      f'validation: {round(train_results.validate_losses[-1], 2)}, ' + 
      f'testing: {round(test_results.test_loss, 2)}' + 
      f'\n(Accuracy) training: {round(100 * train_results.train_accuracies[-1], 2)}%, ' + 
      f'validation: {round(100 * train_results.validate_accuracies[-1], 2)}%, ' + 
      f'testing: {round(100 * test_results.test_accuracy, 2)}%')